# RAG Aquila — Lancement sur Google Colab

Lance les cellules dans l'ordre. Active le GPU : `Runtime > Change runtime type > T4 GPU`

## Étape 1 — Clone du repo GitHub

In [ ]:
from google.colab import userdata
token = userdata.get('AH')

%cd /content
!rm -rf /content/RAG_Aquila
!git clone https://{token}@github.com/paulhenriduranton-collab/RAG_Aquila.git /content/RAG_Aquila
%cd /content/RAG_Aquila

## Étape 2 — Installation des dépendances Python

In [ ]:
!pip install -q langchain-chroma langchain-ollama langchain-core langgraph rank-bm25 sentence-transformers
print('Installation terminée.')

## Étape 3 — Installation d'Ollama et téléchargement des modèles

On utilise Ollama directement sur Colab (même setup qu'en local) pour garantir la compatibilité
des embeddings avec la `vector_db/` versionnée dans le repo.  
Le téléchargement prend ~5-10 min la première fois.

In [ ]:
import time

# Dépendance requise par le script d'installation Ollama
!apt-get install -qq zstd

# Installation d'Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Démarrage du serveur Ollama en arrière-plan
!nohup ollama serve > /tmp/ollama.log 2>&1 &
time.sleep(5)  # Attendre que le serveur soit prêt

# Téléchargement des modèles :
# bge-m3     ~570 Mo — embeddings (ingest + ask)
# gemma2:2b  ~1.6 Go — contextualisation des chunks (ingest)
# gemma4:12b ~7 Go   — HyDE + génération finale (ask)
print('Téléchargement de bge-m3 (~570 Mo)...')
!ollama pull bge-m3

print('Téléchargement de gemma2:2b (~1.6 Go)...')
!ollama pull gemma2:2b

print('Téléchargement de gemma4:12b (~7 Go)...')
!ollama pull gemma4:12b

print('Ollama prêt avec bge-m3, gemma2:2b et gemma4:12b.')

## Étape 4 — Lancement

In [ ]:
!pip install -q ftfy pymupdf4llm docx2txt
!pip install -q langchain-text-splitters
!pip install -q langchain-text-splitters langchain-community
print('Dépendances ingest installées.')

In [ ]:
import subprocess, time, os, sys
from pathlib import Path

# Tuer l ancien serveur Ollama et en relancer un depuis /tmp
# (evite le bug llama-server "cannot get current path" specifique a Colab)
os.system("pkill -f ollama 2>/dev/null; pkill -f llama-server 2>/dev/null")
time.sleep(2)
subprocess.Popen(
    ["ollama", "serve"],
    cwd="/tmp",
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(5)  # Attendre que le serveur soit pret

sys.path.insert(0, "/content/RAG_Aquila/src")
import ingest
ingest.VECTOR_DB_DIR = Path("/content/RAG_Aquila/vector_db")  # ecrit dans le repo clone

ingest.main()

In [ ]:
import subprocess, time, os, sys
from pathlib import Path

# Tuer l ancien serveur Ollama et en relancer un depuis /tmp
# (evite le bug llama-server "cannot get current path" specifique a Colab)
os.system("pkill -f ollama 2>/dev/null; pkill -f llama-server 2>/dev/null")
time.sleep(2)
subprocess.Popen(
    ["ollama", "serve"],
    cwd="/tmp",
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(5)  # Attendre que le serveur soit pret

sys.path.insert(0, "/content/RAG_Aquila/src")
import ask
ask.VECTOR_DB_DIR = Path("/content/RAG_Aquila/vector_db")  # base versionnee dans le repo

import run_agentic_all
run_agentic_all.main()

## Étape 5 — Évaluation RAGAS

Lance cette étape **après** que `run_agentic_all.py` a terminé (étape 4).  
RAGAS évalue automatiquement les 5 métriques clés sur tous les résultats de `data/agentic_results.json`.

In [ ]:
# Installation de ragas + patch compatibilité langchain-community 0.4.x
!pip install -q ragas datasets

# langchain-community 0.4.x a supprimé ChatVertexAI — on patche l'import pour éviter le crash
import site, pathlib

# Cherche le fichier ragas/llms/base.py dans les packages installés
base_py = next(
    (p / "ragas" / "llms" / "base.py"
     for p in map(pathlib.Path, site.getsitepackages())
     if (p / "ragas" / "llms" / "base.py").exists()),
    None
)

if base_py:
    src = base_py.read_text(encoding="utf-8")
    # Applique le patch seulement si l'import direct est encore présent
    old = "from langchain_community.chat_models.vertexai import ChatVertexAI\nfrom langchain_community.llms import VertexAI"
    new = (
        "try:\n"
        "    from langchain_community.chat_models.vertexai import ChatVertexAI\n"
        "    from langchain_community.llms import VertexAI\n"
        "except ImportError:\n"
        "    ChatVertexAI = None\n"
        "    VertexAI = None"
    )
    if old in src:
        base_py.write_text(src.replace(old, new), encoding="utf-8")
        print("Patch appliqué sur ragas/llms/base.py")
    else:
        print("Patch déjà appliqué ou non nécessaire")
else:
    print("ragas/llms/base.py introuvable — vérifier l'installation")

print("Installation prête.")

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "/content/RAG_Aquila/src")

# Import du script d'évaluation
import evaluate_ragas

# Redéfinit les chemins absolus pour Colab
evaluate_ragas.RESULTS_PATH  = Path("/content/RAG_Aquila/data/agentic_results.json")
evaluate_ragas.QUESTIONS_PATH = Path("/content/RAG_Aquila/data/questions.json")
evaluate_ragas.OUTPUT_CSV    = Path("/content/RAG_Aquila/data/ragas_evaluation.csv")

# Utilise gemma2:2b comme juge RAGAS (déjà téléchargé, 3-5x plus rapide que gemma4:12b)
# La qualité de jugement est suffisante pour les métriques oui/non de RAGAS
evaluate_ragas.EVAL_LLM = "gemma2:2b"

evaluate_ragas.main()